# DP Audit Autoresearch — Autonomous Experiment Loop

**EECE 608 — Mohamad Faour**

Runs a Karpathy-style autonomous experiment loop to find stronger membership
inference attacks and maximize `tightness_ratio` (empirical / theoretical epsilon).

An LLM (free via OpenRouter) proposes modifications to the auditing strategy,
the loop runs them, keeps improvements, reverts failures, and learns from history.

### Prerequisites

1. Upload repo as Kaggle Dataset (same one used by `kaggle_full_pipeline.ipynb`)
2. kaggle.com/settings → Secrets → add `OPENROUTER_API_KEY` (free from openrouter.ai/keys)
3. Notebook settings: **Internet ON**, **Persistence: Files**, **CPU** (GPU not needed)
4. Add your dataset + toggle ON the secret in the right sidebar

### How it works

```
LLM proposes new experiment.py → syntax check → run → parse tightness_ratio
  → improved? KEEP : REVERT → log to history → feed history back → repeat
```

---
## 0. Setup

Same setup pattern as `kaggle_full_pipeline.ipynb` — copies the repo to a
writable location since Kaggle input datasets are read-only.

In [ ]:
import shutil, sys, os
from pathlib import Path

# === Point this at your dataset (same as kaggle_full_pipeline.ipynb) ===
INPUT_ROOT = Path("/kaggle/input/eece608-dp-audit/EECE_608")
WORK_ROOT  = Path("/kaggle/working/EECE_608")

# Copy to writable location (autoresearch modifies experiment.py in place)
if not WORK_ROOT.exists():
    shutil.copytree(INPUT_ROOT, WORK_ROOT)
    print(f"Copied to {WORK_ROOT}")
else:
    print(f"Already exists: {WORK_ROOT}")

os.chdir(WORK_ROOT)

# Install deps + project package
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml
!pip install -e . -q

# Add to path so agent_loop can import prepare → dp_audit_tightness
sys.path.insert(0, str(WORK_ROOT / "src"))
sys.path.insert(0, str(WORK_ROOT / "autoresearch"))

print(f"\nReady. Working dir: {os.getcwd()}")

## 1. Resume from previous session (skip if first run)

If you have saved output from a prior session, upload it as a dataset and
uncomment below to restore the 3 state files. The loop will continue where
it left off — it reads `approaches_log.jsonl` to reconstruct full history.

In [ ]:
AR = WORK_ROOT / "autoresearch"

# === UNCOMMENT to restore from a previous session's output dataset ===
# PREV = Path("/kaggle/input/autoresearch-state")  # <-- change to your dataset name
# for fname in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
#     src = PREV / fname
#     if src.exists():
#         shutil.copy2(src, AR / fname)
#         print(f"Restored {fname} ({src.stat().st_size:,} bytes)")

# Show current state
for f in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
    p = AR / f
    status = f"{p.stat().st_size:,} bytes" if p.exists() else "not found (fresh start)"
    print(f"  {f}: {status}")

## 2. Verify baseline

Runs `experiment.py` once to confirm the pipeline works. First run trains and
caches the DP-SGD model (~1 min). After that, each run takes ~10 seconds.

In [ ]:
import subprocess

r = subprocess.run(
    [sys.executable, str(AR / "experiment.py")],
    capture_output=True, text=True, timeout=180, cwd=str(WORK_ROOT),
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-500:])
    raise RuntimeError("Baseline failed — check errors above.")
print("Baseline OK.")

## 3. Run the loop

Imports `run_loop` from the project's `agent_loop.py` and starts the
autonomous cycle. Leave this cell running — it loops until `MAX_EXPERIMENTS`
or until you interrupt it.

**Settings:**
- `MODEL` — free OpenRouter model (Qwen3 Coder 480B is the strongest free option)
- `MAX_EXPERIMENTS` — cap per session (150 ≈ one 9h Kaggle CPU session)
- `TIMEOUT` — seconds per experiment (300 = 5 min)

**What happens each iteration:**
1. Full history (compressed) + current best code → sent to LLM
2. LLM returns modified experiment.py with a hypothesis
3. Syntax pre-check (instant — catches bad code without running it)
4. Run experiment, parse tightness_ratio
5. Keep if improved, revert if not
6. Log to `results.tsv` + `approaches_log.jsonl`

In [ ]:
from kaggle_secrets import UserSecretsClient

# ---- Settings (change these) ----
MODEL           = "qwen/qwen3-coder"     # Best free coder on OpenRouter
MAX_EXPERIMENTS = 150                      # Per session
TIMEOUT         = 300                      # 5 min per experiment
# ---------------------------------

api_key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

# Import the loop directly from the project
from agent_loop import run_loop

best = run_loop(
    api_key=api_key,
    model=MODEL,
    max_experiments=MAX_EXPERIMENTS,
    timeout=TIMEOUT,
)

## 4. Check progress

Run this anytime — even while Cell 3 is still running (open a new code cell below).

In [ ]:
import pandas as pd

tsv = AR / "results.tsv"
if tsv.exists():
    df = pd.read_csv(tsv, sep="\t")
    keeps    = df[df["status"] == "keep"]
    discards = df[df["status"] == "discard"]
    crashes  = df[df["status"].isin(["crash", "timeout"])]

    print(f"Total: {len(df)} | Kept: {len(keeps)} | Discarded: {len(discards)} | Crashed: {len(crashes)}")
    print(f"Best tightness_ratio: {df['tightness_ratio'].max():.6f}\n")
    print("=== Improvements ===")
    print(keeps.to_string(index=False))
    print(f"\n=== Last 5 ===")
    print(df.tail(5).to_string(index=False))
else:
    print("No results yet — run Cell 3 first.")

## 5. Save state (run before session expires)

Copies the 3 state files to `/kaggle/working/` so they appear in the
notebook's **Output** tab. To resume in a new session:

1. Go to this notebook's Output tab
2. Click **New Dataset** to save the output as a dataset (e.g. `autoresearch-state`)
3. In your next notebook, add that dataset and uncomment Cell 1

In [ ]:
OUT = Path("/kaggle/working")

for fname in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
    src = AR / fname
    if src.exists():
        shutil.copy2(src, OUT / fname)
        print(f"Saved {fname} ({src.stat().st_size:,} bytes)")
    else:
        print(f"Not found: {fname}")

print(f"\nSaved to {OUT}/ — check the Output tab.")

## 6. View best experiment code (optional)

In [ ]:
print((AR / "experiment.py").read_text())